<div align="center">
    <img src="../../../media/a365-agents.png" width="100%" alt="Microsoft Foundry workshop / lab / sample"> 
</div>

# Build a Foundry agent that is fully in sync with Agent 365 (WIP!!!)

In this lab you will:

1. **Register an Entra Agent ID** &mdash; a workload identity for your AI agent.
2. **Build a pro-code Foundry agent** with the `azure-ai-projects` SDK and two function tools.
3. **Bind** the Foundry agent to the Entra Agent ID so its runs execute under the agent identity.
4. **Author the Agent 365 manifest** in code (no hidden YAML).
5. **Publish to Agent 365** with the Microsoft 365 Agents Toolkit (`atk`).
6. **Verify the agent is fully in sync** across Entra, Foundry, and Agent 365.
7. **Apply and test five Agent 365 policies**: DLP, Conditional Access, tool allow-list, audit, lifecycle.
8. **Clean up** so re-running the notebook is idempotent.

## Architecture

```
        ┌──────────────────────┐        ┌────────────────────────┐
        │   Entra ID           │        │   Microsoft Foundry    │
        │  (Agent ID = appId)  │◀──bind─│  (Agent runtime)       │
        └──────────┬───────────┘        └────────────┬───────────┘
                   │                                 │
                   │ identity                        │ runtime
                   ▼                                 ▼
        ┌──────────────────────────────────────────────────────────┐
        │                Agent 365 (governance plane)              │
        │  manifest · DLP · Conditional Access · audit · lifecycle │
        └──────────────────────────────────────────────────────────┘
```

**Fully in sync** means the same logical agent is observable in all three planes
with consistent identifiers (`entraAgentId == manifest.identity.entraAgentId`,
`foundryAgentId == manifest.runtime.agentId`).

> **Run the main workshop first.** This lab assumes you completed
> [`src/workshop/README.md`](../../workshop/README.md) and have a populated
> [`src/workshop/.env`](../../workshop/.env).

## Prerequisites

Before running this notebook:

| # | Requirement | How to verify |
|---|-------------|---------------|
| 1 | Main workshop completed | `../../workshop/.env` exists |
| 2 | Azure CLI logged in | `az account show` |
| 3 | Microsoft 365 Agents Toolkit CLI available | `atk --version` |
| 4 | Permission to create/read Entra app registrations | `az ad signed-in-user show` |
| 5 | Permission to upload/install Microsoft 365 app packages | Microsoft 365 tenant policy / admin role |
| 6 | Optional but recommended: Azure Bot Service resource | Needed for an actually responding custom engine agent |
| 7 | Runtime/proxy endpoint | Needed for an actually responding custom engine agent |

The package can validate before the runtime is complete, but the agent will not answer in Copilot/Teams until the Azure Bot endpoint is correctly configured.


## Setup — install lab dependencies

Run this once in the workshop virtual environment.


In [29]:
%pip install -q -r requirements.txt



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 0 — Load environment and define helpers


In [30]:
import json
import os
import sys
import time
import uuid
import zlib
import struct
import hashlib
import subprocess
from pathlib import Path
from urllib.parse import urlparse
import zipfile

import httpx
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load the workshop .env.
WORKSHOP_ENV = Path("../../workshop/.env").resolve()
assert WORKSHOP_ENV.exists(), f"Run the main workshop first — {WORKSHOP_ENV} not found."
load_dotenv(WORKSHOP_ENV)

PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT_NAME = os.environ["AGENT_MODEL_DEPLOYMENT_NAME"]
AZURE_SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
AZURE_RESOURCE_GROUP_NAME = os.environ["AZURE_RESOURCE_GROUP_NAME"]

# Make agent_app.py importable.
sys.path.insert(0, str(Path.cwd()))
import agent_app  # noqa: E402

credential = DefaultAzureCredential()

MANIFEST_DIR = Path.cwd() if Path.cwd().name == "manifest" else Path("manifest")
MANIFEST_DIR.mkdir(exist_ok=True)

def graph_request_raw(method: str, path: str, **kwargs) -> httpx.Response:
    """
    Direct Microsoft Graph v1.0 request helper.

    path examples:
    /applications
    /servicePrincipals
    """
    token = credential.get_token("https://graph.microsoft.com/.default").token

    headers = kwargs.pop("headers", {})
    headers = {
        **headers,
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    if path.startswith("https://"):
        url = path
    else:
        path = path if path.startswith("/") else f"/{path}"
        path = path.removeprefix("/v1.0")
        path = path.removeprefix("/beta")
        url = f"https://graph.microsoft.com/v1.0{path}"

    return httpx.request(method, url, headers=headers, timeout=60, **kwargs)

def arm_request(method: str, path: str, **kwargs) -> httpx.Response:
    """
    Direct Azure Resource Manager request helper.
    """
    token = credential.get_token("https://management.azure.com/.default").token

    headers = kwargs.pop("headers", {})
    headers = {
        **headers,
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    url = path if path.startswith("https://") else f"https://management.azure.com{path}"
    return httpx.request(method, url, headers=headers, timeout=60, **kwargs)

def run_cli(args, check=False, cwd=None):
    """
    Run a CLI command and return CompletedProcess.
    Use check=True only when you want the notebook to fail on errors.
    """
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run(
        [str(a) for a in args],
        cwd=cwd,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(args)}")
    return result

def ensure_guid(value: str, label: str = "value") -> str:
    try:
        return str(uuid.UUID(value))
    except Exception as exc:
        raise ValueError(f"{label} must be a GUID. Got: {value}") from exc

def get_signed_in_user_hint() -> str:
    """
    Best-effort user suffix for deterministic naming.
    Does not fail the notebook if az is unavailable.
    """
    env_user = (
        os.environ.get("USER_PRINCIPAL_NAME")
        or os.environ.get("AZURE_USERNAME")
        or os.environ.get("USERNAME")
        or os.environ.get("USER")
    )
    if env_user:
        return env_user.split("@")[0].replace(".", "-").replace("_", "-")

    result = run_cli(["az", "ad", "signed-in-user", "show", "--query", "userPrincipalName", "-o", "tsv"])
    if result.returncode == 0 and result.stdout.strip():
        return result.stdout.strip().split("@")[0].replace(".", "-").replace("_", "-")

    return "user"

def get_deterministic_agent_name(base_name: str = "foundry-lab-agent") -> str:
    user_hint = get_signed_in_user_hint()
    env_hash = hashlib.sha1(str(WORKSHOP_ENV).encode()).hexdigest()[:6]
    return f"{base_name}-{user_hint}-{env_hash}".lower()

def make_png_rgba(width: int, height: int, rgba: bytes) -> bytes:
    """
    Minimal dependency-free PNG generator.
    """
    raw = b"".join(b"\x00" + rgba * width for _ in range(height))

    def chunk(tag: bytes, data: bytes) -> bytes:
        return (
            struct.pack(">I", len(data))
            + tag
            + data
            + struct.pack(">I", zlib.crc32(tag + data) & 0xFFFFFFFF)
        )

    return (
        b"\x89PNG\r\n\x1a\n"
        + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 6, 0, 0, 0))
        + chunk(b"IDAT", zlib.compress(raw))
        + chunk(b"IEND", b"")
    )

print("[INFO] Foundry endpoint:", PROJECT_ENDPOINT)
print("[INFO] Model deployment:", MODEL_DEPLOYMENT_NAME)
print("[INFO] Azure subscription:", AZURE_SUBSCRIPTION_ID)
print("[INFO] Azure resource group:", AZURE_RESOURCE_GROUP_NAME)
print("[INFO] Manifest directory:", MANIFEST_DIR.resolve())


[INFO] Foundry endpoint: https://aif-aiagents-vbds.services.ai.azure.com/api/projects/workshop-project
[INFO] Model deployment: gpt4o
[INFO] Azure subscription: c200e3e7-0839-483b-848b-25a98451f2cd
[INFO] Azure resource group: rg-kotp-temp
[INFO] Manifest directory: /workspaces/Microsoft-Foundry/src/samples/create-agent365-managed-agents/manifest


## Step 1 — Resolve the identity model

For this custom engine pattern:

- `ENTRA_APP_CLIENT_ID` is the **Application / Client ID** of the Entra App Registration used by the Azure Bot.
- `BOT_ID` is the same value as `ENTRA_APP_CLIENT_ID`.
- The Microsoft 365 manifest uses that bot id in:
  - `bots[0].botId`
  - `copilotAgents.customEngineAgents[0].id`
- The Foundry agent name/version are kept in `foundry-runtime-config.json`.


In [31]:
# You can override these from .env before running the notebook:
# AGENT365_AGENT_BASE_NAME=foundry-lab-agent
# ENTRA_APP_CLIENT_ID=<existing app/client id>
# AZURE_BOT_RESOURCE_NAME=<existing Azure Bot resource name>
# AZURE_BOT_MESSAGING_ENDPOINT=https://<your-host>/api/messages

AGENT_BASE_NAME = os.environ.get("AGENT365_AGENT_BASE_NAME", "foundry-lab-agent")
AGENT_NAME = get_deterministic_agent_name(AGENT_BASE_NAME)

APP_NAME_SHORT = os.environ.get("AGENT365_APP_NAME_SHORT", "Foundry Lab Agent")
APP_NAME_FULL = os.environ.get("AGENT365_APP_NAME_FULL", "Foundry Lab Agent for Microsoft 365")
APP_VERSION = os.environ.get("AGENT365_APP_VERSION", "1.0.0")

DEVELOPER_NAME = os.environ.get("AGENT365_DEVELOPER_NAME", "Douwe van de Ruit")
WEBSITE_URL = os.environ.get("AGENT365_WEBSITE_URL", "https://example.com")
PRIVACY_URL = os.environ.get("AGENT365_PRIVACY_URL", "https://example.com/privacy")
TERMS_URL = os.environ.get("AGENT365_TERMS_URL", "https://example.com/terms")

AZURE_BOT_RESOURCE_GROUP = os.environ.get("AZURE_BOT_RESOURCE_GROUP", AZURE_RESOURCE_GROUP_NAME)
AZURE_BOT_RESOURCE_NAME = os.environ.get("AZURE_BOT_RESOURCE_NAME", AGENT_NAME)
AZURE_BOT_MESSAGING_ENDPOINT = os.environ.get(
    "AZURE_BOT_MESSAGING_ENDPOINT",
    "https://example.com/api/messages",
)

ENABLE_WEB_APPLICATION_INFO = os.environ.get("ENABLE_WEB_APPLICATION_INFO", "false").lower() == "true"
CREATE_AZURE_BOT_IF_MISSING = os.environ.get("CREATE_AZURE_BOT_IF_MISSING", "false").lower() == "true"
RUN_ATK_VALIDATE = os.environ.get("RUN_ATK_VALIDATE", "true").lower() == "true"
RUN_ATK_INSTALL = os.environ.get("RUN_ATK_INSTALL", "false").lower() == "true"

print("[INFO] AGENT_NAME:", AGENT_NAME)
print("[INFO] APP_NAME_SHORT:", APP_NAME_SHORT)
print("[INFO] AZURE_BOT_RESOURCE_NAME:", AZURE_BOT_RESOURCE_NAME)
print("[INFO] AZURE_BOT_MESSAGING_ENDPOINT:", AZURE_BOT_MESSAGING_ENDPOINT)
print("[INFO] ENABLE_WEB_APPLICATION_INFO:", ENABLE_WEB_APPLICATION_INFO)
print("[INFO] CREATE_AZURE_BOT_IF_MISSING:", CREATE_AZURE_BOT_IF_MISSING)


[INFO] AGENT_NAME: foundry-lab-agent-vscode-dd3ae0
[INFO] APP_NAME_SHORT: Foundry Lab Agent
[INFO] AZURE_BOT_RESOURCE_NAME: foundry-lab-agent-vscode-dd3ae0
[INFO] AZURE_BOT_MESSAGING_ENDPOINT: https://example.com/api/messages
[INFO] ENABLE_WEB_APPLICATION_INFO: False
[INFO] CREATE_AZURE_BOT_IF_MISSING: False


## Step 2 — Create or reuse the Entra App Registration used by the Azure Bot

This creates/reuses a normal Entra App Registration. It is used as the Bot Framework / Azure Bot identity.

If you already have one, set `ENTRA_APP_CLIENT_ID` in `.env` and this step will reuse it.


In [32]:
def find_application_by_app_id(app_id: str):
    response = graph_request_raw(
        "GET",
        "/applications",
        params={
            "$filter": f"appId eq '{app_id}'",
            "$select": "id,appId,displayName",
        },
    )
    response.raise_for_status()
    values = response.json().get("value", [])
    return values[0] if values else None

def find_application_by_display_name(display_name: str):
    response = graph_request_raw(
        "GET",
        "/applications",
        params={
            "$filter": f"displayName eq '{display_name}'",
            "$select": "id,appId,displayName",
        },
    )
    response.raise_for_status()
    values = response.json().get("value", [])
    return values[0] if values else None

def create_application(display_name: str):
    response = graph_request_raw(
        "POST",
        "/applications",
        json={"displayName": display_name},
    )
    if response.status_code not in (200, 201):
        raise RuntimeError(
            "Could not create Entra application.\n"
            f"Status: {response.status_code}\n"
            f"Body: {response.text}"
        )
    return response.json()

def get_or_create_service_principal(app_id: str):
    response = graph_request_raw(
        "GET",
        "/servicePrincipals",
        params={
            "$filter": f"appId eq '{app_id}'",
            "$select": "id,appId,displayName",
        },
    )
    response.raise_for_status()
    values = response.json().get("value", [])
    if values:
        return values[0]

    response = graph_request_raw("POST", "/servicePrincipals", json={"appId": app_id})
    if response.status_code not in (200, 201):
        raise RuntimeError(
            "Could not create service principal.\n"
            f"Status: {response.status_code}\n"
            f"Body: {response.text}"
        )
    return response.json()

existing_client_id = os.environ.get("ENTRA_APP_CLIENT_ID") or os.environ.get("ENTRA_AGENT_ID")

if existing_client_id:
    ENTRA_APP_CLIENT_ID = ensure_guid(existing_client_id, "ENTRA_APP_CLIENT_ID")
    app = find_application_by_app_id(ENTRA_APP_CLIENT_ID)
    assert app, f"No Entra application found for ENTRA_APP_CLIENT_ID={ENTRA_APP_CLIENT_ID}"
    print("[INFO] Reusing Entra application from ENTRA_APP_CLIENT_ID.")
else:
    app = find_application_by_display_name(AGENT_NAME)
    if app:
        print("[INFO] Reusing existing Entra application by displayName.")
    else:
        print("[INFO] Creating new Entra application.")
        app = create_application(AGENT_NAME)

ENTRA_APP_OBJECT_ID = app["id"]
ENTRA_APP_CLIENT_ID = app["appId"]
BOT_ID = ensure_guid(ENTRA_APP_CLIENT_ID, "BOT_ID")

service_principal = get_or_create_service_principal(BOT_ID)
ENTRA_SERVICE_PRINCIPAL_OBJECT_ID = service_principal["id"]

print("[INFO] Entra application display name:", app.get("displayName"))
print("[INFO] Entra app object id:", ENTRA_APP_OBJECT_ID)
print("[INFO] Entra app/client id:", ENTRA_APP_CLIENT_ID)
print("[INFO] Service principal object id:", ENTRA_SERVICE_PRINCIPAL_OBJECT_ID)
print("[INFO] BOT_ID:", BOT_ID)


[INFO] Creating new Entra application.
[INFO] Entra application display name: foundry-lab-agent-vscode-dd3ae0
[INFO] Entra app object id: 01d22cd4-e5b8-469b-ab9f-3f9c50c3cce9
[INFO] Entra app/client id: feee351c-0c81-42a6-8a9c-ead17f8c2fc3
[INFO] Service principal object id: 7a6f0aaa-10e8-4082-b5d1-81b9ad32c6df
[INFO] BOT_ID: feee351c-0c81-42a6-8a9c-ead17f8c2fc3


## Step 3 — Grant Azure RBAC to the app/service principal

The bot/proxy or runtime may need access to the Foundry project or related Azure resources. This cell grants the service principal the `Cognitive Services User` role at the resource group scope.

Adjust `ROLE_NAME` and `rbac_scope` if your Foundry project requires a more specific scope or role.


In [33]:
ROLE_NAME = os.environ.get("AGENT365_RBAC_ROLE_NAME", "Cognitive Services User")

rbac_scope = (
    f"/subscriptions/{AZURE_SUBSCRIPTION_ID}"
    f"/resourceGroups/{AZURE_RESOURCE_GROUP_NAME}"
)

role_defs_response = arm_request(
    "GET",
    f"{rbac_scope}/providers/Microsoft.Authorization/roleDefinitions",
    params={
        "api-version": "2022-04-01",
        "$filter": f"roleName eq '{ROLE_NAME}'",
    },
)

if role_defs_response.status_code >= 400:
    raise RuntimeError(
        "Failed to query Azure role definitions.\n"
        f"Status: {role_defs_response.status_code}\n"
        f"URL: {role_defs_response.request.url}\n"
        f"Body: {role_defs_response.text}"
    )

role_defs = role_defs_response.json().get("value", [])
if not role_defs:
    raise RuntimeError(f"Azure RBAC role not found: {ROLE_NAME}")

role_definition_id = role_defs[0]["id"]

assignment_id = str(
    uuid.uuid5(
        uuid.NAMESPACE_URL,
        f"{rbac_scope}|{role_definition_id}|{ENTRA_SERVICE_PRINCIPAL_OBJECT_ID}",
    )
)

assignment_path = (
    f"{rbac_scope}/providers/Microsoft.Authorization"
    f"/roleAssignments/{assignment_id}"
)

payload = {
    "properties": {
        "roleDefinitionId": role_definition_id,
        "principalId": ENTRA_SERVICE_PRINCIPAL_OBJECT_ID,
        "principalType": "ServicePrincipal",
    }
}

for attempt in range(1, 6):
    response = arm_request(
        "PUT",
        assignment_path,
        params={"api-version": "2022-04-01"},
        json=payload,
    )

    if response.status_code in (200, 201):
        print(f"[INFO] Role assignment OK: {ROLE_NAME}")
        print(f"[INFO] Principal: {ENTRA_SERVICE_PRINCIPAL_OBJECT_ID}")
        print(f"[INFO] Scope: {rbac_scope}")
        break

    if response.status_code == 409:
        print(f"[INFO] Role assignment already exists: {ROLE_NAME}")
        print(f"[INFO] Principal: {ENTRA_SERVICE_PRINCIPAL_OBJECT_ID}")
        print(f"[INFO] Scope: {rbac_scope}")
        break

    if "PrincipalNotFound" in response.text and attempt < 5:
        print("[WARN] Principal not visible to Azure RBAC yet. Retrying...")
        time.sleep(10)
        continue

    raise RuntimeError(
        "Role assignment failed.\n"
        f"Status: {response.status_code}\n"
        f"URL: {response.request.url}\n"
        f"Body: {response.text}"
    )


[INFO] Role assignment OK: Cognitive Services User
[INFO] Principal: 7a6f0aaa-10e8-4082-b5d1-81b9ad32c6df
[INFO] Scope: /subscriptions/c200e3e7-0839-483b-848b-25a98451f2cd/resourceGroups/rg-kotp-temp


## Step 4 — Build or reuse the Foundry agent

The Foundry agent is the runtime object your backend/proxy will call.

This notebook does **not** put the Foundry agent id into the Microsoft 365 manifest. Instead, it writes a runtime config file that your backend/proxy can use.


In [34]:
from agent_app import build_or_get_agent

project_client = AIProjectClient(PROJECT_ENDPOINT, credential)

agent_handle = build_or_get_agent(
    project_client=project_client,
    model_deployment_name=MODEL_DEPLOYMENT_NAME,
    agent_name=AGENT_NAME,
)

def first_existing_attr(obj, attr_names, default=None):
    for attr in attr_names:
        if hasattr(obj, attr):
            value = getattr(obj, attr)
            if value is not None:
                return value
    return default

FOUNDRY_AGENT_ID = first_existing_attr(
    agent_handle,
    ["id", "agent_id", "assistant_id", "name"],
    default=AGENT_NAME,
)

FOUNDRY_AGENT_NAME = first_existing_attr(agent_handle, ["name"], default=AGENT_NAME)
FOUNDRY_AGENT_VERSION = first_existing_attr(agent_handle, ["version"], default=None)

print("[INFO] Foundry agent name:", FOUNDRY_AGENT_NAME)
print("[INFO] Foundry agent id/name handle:", FOUNDRY_AGENT_ID)
print("[INFO] Foundry agent version:", FOUNDRY_AGENT_VERSION)


[INFO] Foundry agent name: foundry-lab-agent-vscode-dd3ae0
[INFO] Foundry agent id/name handle: foundry-lab-agent-vscode-dd3ae0
[INFO] Foundry agent version: 1


## Step 5 — Write the Foundry runtime config

This file is for your backend/proxy runtime. It is **not** part of the Microsoft 365 app package.

Your bot/proxy should use this file or equivalent environment variables to route incoming Microsoft 365/Bot Framework traffic to the correct Foundry project and agent.


In [35]:
runtime_config = {
    "m365": {
        "appManifestId": BOT_ID,
        "appName": APP_NAME_SHORT,
        "manifestVersion": APP_VERSION,
    },
    "bot": {
        "botId": BOT_ID,
        "entraAppClientId": ENTRA_APP_CLIENT_ID,
        "entraAppObjectId": ENTRA_APP_OBJECT_ID,
        "servicePrincipalObjectId": ENTRA_SERVICE_PRINCIPAL_OBJECT_ID,
        "azureBotResourceGroup": AZURE_BOT_RESOURCE_GROUP,
        "azureBotResourceName": AZURE_BOT_RESOURCE_NAME,
        "messagingEndpoint": AZURE_BOT_MESSAGING_ENDPOINT,
    },
    "foundry": {
        "projectEndpoint": PROJECT_ENDPOINT,
        "modelDeploymentName": MODEL_DEPLOYMENT_NAME,
        "agentName": FOUNDRY_AGENT_NAME,
        "agentIdOrHandle": FOUNDRY_AGENT_ID,
        "agentVersion": FOUNDRY_AGENT_VERSION,
    },
    "notes": [
        "The Microsoft 365 manifest points to the bot id.",
        "The bot/proxy runtime points to the Foundry agent.",
        "Do not add non-standard identity/runtime fields to manifest.json.",
    ],
}

runtime_config_path = MANIFEST_DIR / "foundry-runtime-config.json"
with open(runtime_config_path, "w", encoding="utf-8") as f:
    json.dump(runtime_config, f, indent=2)

print(f"[INFO] Runtime config written to: {runtime_config_path.resolve()}")
print(json.dumps(runtime_config, indent=2))


[INFO] Runtime config written to: /workspaces/Microsoft-Foundry/src/samples/create-agent365-managed-agents/manifest/foundry-runtime-config.json
{
  "m365": {
    "appManifestId": "feee351c-0c81-42a6-8a9c-ead17f8c2fc3",
    "appName": "Foundry Lab Agent",
    "manifestVersion": "1.0.0"
  },
  "bot": {
    "botId": "feee351c-0c81-42a6-8a9c-ead17f8c2fc3",
    "entraAppClientId": "feee351c-0c81-42a6-8a9c-ead17f8c2fc3",
    "entraAppObjectId": "01d22cd4-e5b8-469b-ab9f-3f9c50c3cce9",
    "servicePrincipalObjectId": "7a6f0aaa-10e8-4082-b5d1-81b9ad32c6df",
    "azureBotResourceGroup": "rg-kotp-temp",
    "azureBotResourceName": "foundry-lab-agent-vscode-dd3ae0",
    "messagingEndpoint": "https://example.com/api/messages"
  },
  "foundry": {
    "projectEndpoint": "https://aif-aiagents-vbds.services.ai.azure.com/api/projects/workshop-project",
    "modelDeploymentName": "gpt4o",
    "agentName": "foundry-lab-agent-vscode-dd3ae0",
    "agentIdOrHandle": "foundry-lab-agent-vscode-dd3ae0",
    "ag

## Step 6 — Check or create the Azure Bot resource

A custom engine agent needs a bot identity and a reachable HTTPS endpoint to actually respond.

This cell checks whether an Azure Bot resource exists. It only creates one if:

```python
CREATE_AZURE_BOT_IF_MISSING = True
```

and the endpoint is not the placeholder `https://example.com/api/messages`.

If you are only validating Agent 365 registration/governance, you can continue without creating the bot here. For a working runtime demo, configure the bot endpoint to your proxy service.


In [36]:
def get_azure_bot_resource(resource_group: str, name: str):
    result = run_cli([
        "az", "bot", "show",
        "--resource-group", resource_group,
        "--name", name,
        "--query", "{name:name, appId:properties.msaAppId, endpoint:properties.endpoint}",
        "-o", "json",
    ])
    if result.returncode != 0:
        return None
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return None

azure_bot = get_azure_bot_resource(AZURE_BOT_RESOURCE_GROUP, AZURE_BOT_RESOURCE_NAME)

if azure_bot:
    print("[INFO] Azure Bot resource found:")
    print(json.dumps(azure_bot, indent=2))
else:
    print("[WARN] Azure Bot resource was not found.")
    print("[INFO] Resource group:", AZURE_BOT_RESOURCE_GROUP)
    print("[INFO] Bot resource name:", AZURE_BOT_RESOURCE_NAME)

    if CREATE_AZURE_BOT_IF_MISSING:
        if "example.com" in AZURE_BOT_MESSAGING_ENDPOINT:
            raise RuntimeError(
                "CREATE_AZURE_BOT_IF_MISSING=true, but AZURE_BOT_MESSAGING_ENDPOINT is still a placeholder."
            )

        print("[INFO] Creating Azure Bot registration.")
        create_result = run_cli([
            "az", "bot", "create",
            "--resource-group", AZURE_BOT_RESOURCE_GROUP,
            "--name", AZURE_BOT_RESOURCE_NAME,
            "--kind", "registration",
            "--appid", BOT_ID,
            "--endpoint", AZURE_BOT_MESSAGING_ENDPOINT,
        ])

        if create_result.returncode != 0:
            raise RuntimeError("Azure Bot creation failed. See output above.")

        azure_bot = get_azure_bot_resource(AZURE_BOT_RESOURCE_GROUP, AZURE_BOT_RESOURCE_NAME)
    else:
        print("[INFO] Skipping Azure Bot creation. Set CREATE_AZURE_BOT_IF_MISSING=true to create it.")

if azure_bot:
    assert azure_bot.get("appId") == BOT_ID, (
        "Azure Bot appId does not match BOT_ID. "
        f"Azure Bot appId={azure_bot.get('appId')} BOT_ID={BOT_ID}"
    )
    if not azure_bot.get("endpoint"):
        print("[WARN] Azure Bot exists but has no endpoint configured.")
    print("[SUCCESS] Azure Bot identity is aligned with the manifest BOT_ID.")
else:
    print("[WARN] No Azure Bot resource verified. Manifest can be generated, but runtime linkage is incomplete.")


$ az bot show --resource-group rg-kotp-temp --name foundry-lab-agent-vscode-dd3ae0 --query {name:name, appId:properties.msaAppId, endpoint:properties.endpoint} -o json
ERROR: ResourceNotFoundError: (ResourceGroupNotFound) Resource group 'rg-kotp-temp' could not be found.

[WARN] Azure Bot resource was not found.
[INFO] Resource group: rg-kotp-temp
[INFO] Bot resource name: foundry-lab-agent-vscode-dd3ae0
[INFO] Skipping Azure Bot creation. Set CREATE_AZURE_BOT_IF_MISSING=true to create it.
[WARN] No Azure Bot resource verified. Manifest can be generated, but runtime linkage is incomplete.


## Step 7 — Generate the Microsoft 365 custom engine agent manifest

This is the key fix.

The generated package uses:

- `manifestVersion: 1.21`
- `copilotAgents.customEngineAgents`
- `bots`
- no `packageName`
- no `declarativeAgent.json`
- no non-standard `identity` or `runtime` fields


In [37]:
def domain_from_url(url: str):
    parsed = urlparse(url)
    if parsed.netloc:
        return parsed.netloc
    return None

valid_domains = {
    domain_from_url(WEBSITE_URL),
    domain_from_url(PRIVACY_URL),
    domain_from_url(TERMS_URL),
    domain_from_url(AZURE_BOT_MESSAGING_ENDPOINT),
}
valid_domains = sorted(d for d in valid_domains if d and d != "example.com")
if not valid_domains:
    valid_domains = ["example.com"]

manifest = {
    "$schema": "https://developer.microsoft.com/json-schemas/teams/v1.21/MicrosoftTeams.schema.json",
    "manifestVersion": "1.21",
    "version": APP_VERSION,

    # For this lab we use the bot/app client id as the app manifest id as well.
    # Production apps may choose to keep the M365 app id and bot id separate,
    # but the manifest must still keep bots[0].botId == customEngineAgents[0].id.
    "id": BOT_ID,

    "developer": {
        "name": DEVELOPER_NAME,
        "websiteUrl": WEBSITE_URL,
        "privacyUrl": PRIVACY_URL,
        "termsOfUseUrl": TERMS_URL,
    },

    "name": {
        "short": APP_NAME_SHORT,
        "full": APP_NAME_FULL,
    },

    "description": {
        "short": "Foundry-backed custom engine agent for Microsoft 365.",
        "full": (
            "A Microsoft 365 custom engine agent that routes interactions through "
            "an Azure Bot / proxy runtime to a Microsoft Foundry agent. "
            "This package is intended to demonstrate Agent 365 registration, "
            "governance, access controls, restrictions, and policy behavior."
        ),
    },

    "icons": {
        "color": "color.png",
        "outline": "outline.png",
    },

    "accentColor": "#FFFFFF",
    "validDomains": valid_domains,

    "copilotAgents": {
        "customEngineAgents": [
            {
                "type": "bot",
                "id": BOT_ID,
                "disclaimer": {
                    "text": "This agent uses a custom runtime connected to Microsoft Foundry."
                },
            }
        ]
    },

    "bots": [
        {
            "botId": BOT_ID,
            "scopes": [
                "copilot",
                "personal",
                "team",
            ],
            "supportsFiles": False,
            "isNotificationOnly": False,
            "commandLists": [
                {
                    "scopes": [
                        "copilot",
                        "personal",
                    ],
                    "commands": [
                        {
                            "title": "How can you help me?",
                            "description": "Explain what this Foundry-backed custom engine agent can do.",
                        },
                        {
                            "title": "Show governance status",
                            "description": "Explain how this agent is registered and governed in Agent 365.",
                        },
                    ],
                }
            ],
        }
    ],
}

if ENABLE_WEB_APPLICATION_INFO:
    manifest["webApplicationInfo"] = {
        "id": BOT_ID,
        "resource": os.environ.get("AGENT365_APPLICATION_ID_URI", f"api://{BOT_ID}"),
    }

# Required icons.
# color.png must be 192x192 without transparency.
# outline.png must be 32x32 white/transparent.
(MANIFEST_DIR / "color.png").write_bytes(
    make_png_rgba(192, 192, b"\xff\xff\xff\xff")
)

(MANIFEST_DIR / "outline.png").write_bytes(
    make_png_rgba(32, 32, b"\xff\xff\xff\x00")
)

manifest_path = MANIFEST_DIR / "manifest.json"
zip_path = MANIFEST_DIR / "appPackage.zip"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in ["manifest.json", "color.png", "outline.png"]:
        zf.write(MANIFEST_DIR / fname, fname)

print(f"[INFO] manifest.json written to: {manifest_path.resolve()}")
print(f"[INFO] appPackage.zip written to: {zip_path.resolve()}")
print(f"[INFO] Manifest id: {manifest['id']}")
print(f"[INFO] Bot id: {manifest['bots'][0]['botId']}")
print(f"[INFO] Custom engine id: {manifest['copilotAgents']['customEngineAgents'][0]['id']}")
print(f"[INFO] validDomains: {manifest['validDomains']}")


[INFO] manifest.json written to: /workspaces/Microsoft-Foundry/src/samples/create-agent365-managed-agents/manifest/manifest.json
[INFO] appPackage.zip written to: /workspaces/Microsoft-Foundry/src/samples/create-agent365-managed-agents/manifest/appPackage.zip
[INFO] Manifest id: feee351c-0c81-42a6-8a9c-ead17f8c2fc3
[INFO] Bot id: feee351c-0c81-42a6-8a9c-ead17f8c2fc3
[INFO] Custom engine id: feee351c-0c81-42a6-8a9c-ead17f8c2fc3
[INFO] validDomains: ['example.com']


## Step 8 — Inspect package before validation


In [38]:
with zipfile.ZipFile(zip_path, "r") as zf:
    print("[INFO] Package contents:")
    for item in zf.namelist():
        print(" -", item)

with open(manifest_path, "r", encoding="utf-8") as f:
    generated_manifest = json.load(f)

print("\n[INFO] Key manifest fields:")
print(json.dumps({
    "manifestVersion": generated_manifest.get("manifestVersion"),
    "id": generated_manifest.get("id"),
    "version": generated_manifest.get("version"),
    "name": generated_manifest.get("name"),
    "copilotAgents": generated_manifest.get("copilotAgents"),
    "bots": generated_manifest.get("bots"),
    "packageName": generated_manifest.get("packageName"),
    "webApplicationInfo": generated_manifest.get("webApplicationInfo"),
}, indent=2))

assert "packageName" not in generated_manifest, "packageName must not be present."
assert "declarativeAgents" not in generated_manifest.get("copilotAgents", {}), (
    "This lab targets custom engine agents, not declarative agents."
)
assert generated_manifest["bots"][0]["botId"] == generated_manifest["copilotAgents"]["customEngineAgents"][0]["id"]
assert "copilot" in generated_manifest["bots"][0]["scopes"]
assert "personal" in generated_manifest["bots"][0]["scopes"]

print("[SUCCESS] Package structure and manifest wiring look correct.")


[INFO] Package contents:
 - manifest.json
 - color.png
 - outline.png

[INFO] Key manifest fields:
{
  "manifestVersion": "1.21",
  "id": "feee351c-0c81-42a6-8a9c-ead17f8c2fc3",
  "version": "1.0.0",
  "name": {
    "short": "Foundry Lab Agent",
    "full": "Foundry Lab Agent for Microsoft 365"
  },
  "copilotAgents": {
    "customEngineAgents": [
      {
        "type": "bot",
        "id": "feee351c-0c81-42a6-8a9c-ead17f8c2fc3",
        "disclaimer": {
          "text": "This agent uses a custom runtime connected to Microsoft Foundry."
        }
      }
    ]
  },
  "bots": [
    {
      "botId": "feee351c-0c81-42a6-8a9c-ead17f8c2fc3",
      "scopes": [
        "copilot",
        "personal",
        "team"
      ],
      "supportsFiles": false,
      "isNotificationOnly": false,
      "commandLists": [
        {
          "scopes": [
            "copilot",
            "personal"
          ],
          "commands": [
            {
              "title": "How can you help me?",
        

## Publish and install the Agent 365 app package (manual)

Use the **Microsoft 365 Agents Toolkit CLI** (`atk`) to validate and install the manifest generated above into your tenant.

**Steps:**
1. Download the file `appPackage.zip` from the `manifest` folder to your local machine.
2. Open a terminal on your own laptop or use the Azure Cloud shell. 
3. Run the following commands:

```bash
atk validate --package-file appPackage.zip
atk install --file-path appPackage.zip --scope Shared
```

- You must be signed in to your Microsoft 365 tenant with `atk auth login m365`.
- Follow the CLI instructions to sign in.
- After installation, the agent will appear in the Microsoft 365 admin center under **Agents**.

Confirm the agent appears in the **Microsoft 365 admin center**:

1. Open <https://admin.microsoft.com> → **Agents**.
2. You should see **Foundry Lab Agent** with status *Published*.
3. Note the **Agent 365 manifest id** in the details panel — you will use it in the next step.

Using the agent in the Copilot interface:

1. In the terminal where you executed the atk install command the output should include a link:

<img src="../../../media/a365-term.png" width="100%" alt="terminal output"> 

2. Copy and paste the URL into your browser. You should be able to use your agent:

<img src="../../../media/a365-chat.png" width="100%" alt="chat"> 

<hr />

TROUBLESHOOTING: 
If you run into an error due to versioning, consider bumping the version numbers or removing the old package by using the following commands:

```bash
MANIFEST_ID=$(unzip -p appPackage.zip manifest.json | jq -r '.id')

atk uninstall \
  --mode manifest-id \
  --manifest-id "$MANIFEST_ID" \
  --options "m365-app" \
  --interactive false
```

## Step 10 — Verify sync across Microsoft 365 manifest, Entra/Bot identity, Azure Bot, and Foundry config



In [42]:
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest_check = json.load(f)

with open(runtime_config_path, "r", encoding="utf-8") as f:
    runtime_check = json.load(f)

manifest_id = manifest_check["id"]
manifest_bot_id = manifest_check["bots"][0]["botId"]
custom_engine_id = manifest_check["copilotAgents"]["customEngineAgents"][0]["id"]

print("[VERIFY] Manifest id:", manifest_id)
print("[VERIFY] Manifest bot id:", manifest_bot_id)
print("[VERIFY] Custom engine agent id:", custom_engine_id)
print("[VERIFY] Entra app/client id:", ENTRA_APP_CLIENT_ID)
print("[VERIFY] Foundry agent:", runtime_check["foundry"]["agentName"])

assert ensure_guid(manifest_id, "manifest.id") == manifest_id
assert manifest_bot_id == custom_engine_id, "bots[0].botId must match customEngineAgents[0].id"
assert manifest_bot_id == BOT_ID, "manifest bot id must match BOT_ID"
assert runtime_check["bot"]["botId"] == BOT_ID, "runtime config bot id mismatch"
assert runtime_check["foundry"]["projectEndpoint"] == PROJECT_ENDPOINT, "Foundry endpoint mismatch"
assert runtime_check["foundry"]["modelDeploymentName"] == MODEL_DEPLOYMENT_NAME, "Model deployment mismatch"
assert runtime_check["foundry"]["agentName"] == FOUNDRY_AGENT_NAME, "Foundry agent name mismatch"

app_check = find_application_by_app_id(BOT_ID)
assert app_check, "Entra application not found for BOT_ID"
print("[SUCCESS] Entra application exists:", app_check.get("displayName"))

sp_check = get_or_create_service_principal(BOT_ID)
assert sp_check, "Service principal not found for BOT_ID"
print("[SUCCESS] Service principal exists:", sp_check.get("id"))

azure_bot_check = get_azure_bot_resource(AZURE_BOT_RESOURCE_GROUP, AZURE_BOT_RESOURCE_NAME)
if azure_bot_check:
    print("[SUCCESS] Azure Bot resource found:")
    print(json.dumps(azure_bot_check, indent=2))
    assert azure_bot_check.get("appId") == BOT_ID, "Azure Bot appId mismatch"
else:
    print("[WARN] Azure Bot resource not found or not readable.")
    print("[WARN] Agent 365 registration may still validate, but runtime linkage is incomplete.")

print("[SUCCESS] Verification completed without using non-standard manifest identity/runtime fields.")


[VERIFY] Manifest id: feee351c-0c81-42a6-8a9c-ead17f8c2fc3
[VERIFY] Manifest bot id: feee351c-0c81-42a6-8a9c-ead17f8c2fc3
[VERIFY] Custom engine agent id: feee351c-0c81-42a6-8a9c-ead17f8c2fc3
[VERIFY] Entra app/client id: feee351c-0c81-42a6-8a9c-ead17f8c2fc3
[VERIFY] Foundry agent: foundry-lab-agent-vscode-dd3ae0
[SUCCESS] Entra application exists: foundry-lab-agent-vscode-dd3ae0
[SUCCESS] Service principal exists: 7a6f0aaa-10e8-4082-b5d1-81b9ad32c6df
$ az bot show --resource-group rg-kotp-temp --name foundry-lab-agent-vscode-dd3ae0 --query {name:name, appId:properties.msaAppId, endpoint:properties.endpoint} -o json
ERROR: ResourceNotFoundError: (ResourceNotFound) The Resource 'Microsoft.BotService/botServices/foundry-lab-agent-vscode-dd3ae0' under resource group 'rg-kotp-temp' was not found. For more details please go to https://aka.ms/ARMResourceNotFoundFix

[WARN] Azure Bot resource not found or not readable.
[WARN] Agent 365 registration may still validate, but runtime linkage is i

## Step 11 — Confirm in Microsoft 365 admin center

After install:

1. Open `https://admin.microsoft.com`
2. Go to **Agents**
3. Find the agent by name, for example `Foundry Lab Agent`
4. Confirm it appears as an organization/custom agent
5. Open the details pane and note:
   - Manifest id
   - Publisher/developer
   - Availability
   - User/group assignment
   - Status
   - Policy or restriction state

For the governance demo, use the admin center to show:

| Demo | What to show |
|------|--------------|
| Discovery | The agent appears in the Agent inventory |
| Availability | Restrict agent availability to selected users/groups |
| Disable/enable | Block and unblock the agent |
| Lifecycle | Version bump and re-install/update |
| Runtime separation | Agent 365 governs the M365 agent identity, while the backend routes to Foundry |


## Step 12 — Policy demo checklist

Use this section as the script for your demo. These are portal/runtime checks, not manifest fields.

| Policy area | Demo action | Expected effect |
|-------------|-------------|-----------------|
| Access policy | Limit availability to a test group | Users outside the group cannot use/see the agent |
| Block policy | Disable the agent | Agent becomes unavailable |
| Conditional Access | Target the Entra app/service principal or user access path depending on tenant policy model | Access follows tenant rules |
| DLP / data boundary | Route sensitive prompt through runtime and show policy behavior | Runtime/tenant controls determine allowed flow |
| Audit | Invoke the agent, then inspect audit/usage surfaces | Invocation should be attributable to the app/agent path |
| Tool allow-list | Disable or restrict backend actions in your proxy | The agent remains registered, but restricted action fails safely |

The manifest is only the registration surface. Actual tool restriction and Foundry routing must be enforced in your bot/proxy runtime.


## Step 13 — Optional cleanup

This lab does not delete cloud resources by default.

Set `RUN_CLEANUP_LOCAL_ONLY = True` if you only want to remove generated local files. Do not delete Entra/Bot/Foundry resources unless you are sure they are not used elsewhere.


In [ ]:
RUN_CLEANUP_LOCAL_ONLY = os.environ.get("RUN_CLEANUP_LOCAL_ONLY", "false").lower() == "true"

if RUN_CLEANUP_LOCAL_ONLY:
    for file_name in [
        "manifest.json",
        "appPackage.zip",
        "color.png",
        "outline.png",
        "foundry-runtime-config.json",
    ]:
        file_path = MANIFEST_DIR / file_name
        if file_path.exists():
            file_path.unlink()
            print("[INFO] Deleted:", file_path)
    print("[SUCCESS] Local generated files removed.")
else:
    print("[INFO] RUN_CLEANUP_LOCAL_ONLY is false. No cleanup performed.")
